In [34]:
import pandas as pd
import sqlite3

# 物理格式化内存宇宙
if 'conn' in locals():
    conn.close()
conn = sqlite3.connect(':memory:')

# 🛰️ 极端交错：状态变更流（Status Log）
status_data = {
    'user_id':     [701,                     701,                     702],
    'status':      [' VIP_START ',           ' RISK_LOCK ',           ' VIP_START '], 
    'status_time': ['2026-06-09 10:00:00',   '2026-06-09 10:15:00',   '2026-06-09 11:00:00'] # 701 在15分钟内状态流转
}
df_status = pd.DataFrame(status_data)
df_status.to_sql('status_log', conn, index=False, if_exists='replace')

# ⚡ 密集行为流（Behavior Log）
behavior_data = {
    'action_id':     [1001,  1002,  1003,  1004],
    'user_id':       [701,   701,   701,   702],
    'behavior':      ['click', 'buy', 'click', 'buy'],
    'behavior_time': [
        '2026-06-09 10:05:00', # 701 行为 A -> 应该归属于 VIP_START
        '2026-06-09 10:20:00', # 701 行为 B -> 虽然距VIP未超30m，但VIP已被砍断！必须归属于 RISK_LOCK
        '2026-06-09 10:40:00', # 701 行为 C -> 属于 RISK_LOCK 的 30 分钟时效内
        '2026-06-09 11:15:00'  # 702 行为 D -> 属于 VIP_START
    ]
}
df_behavior = pd.DataFrame(behavior_data)
df_behavior.to_sql('behavior_log', conn, index=False, if_exists='replace')

print("=== 🌋 极端时空窗口测试沙盒已完成物理注入！ ===")
print("状态变更表status_log:")
print(df_status)
print("用户高频行为表behavior_log:")
print(df_behavior)

=== 🌋 极端时空窗口测试沙盒已完成物理注入！ ===
状态变更表status_log:
   user_id       status          status_time
0      701   VIP_START   2026-06-09 10:00:00
1      701   RISK_LOCK   2026-06-09 10:15:00
2      702   VIP_START   2026-06-09 11:00:00
用户高频行为表behavior_log:
   action_id  user_id behavior        behavior_time
0       1001      701    click  2026-06-09 10:05:00
1       1002      701      buy  2026-06-09 10:20:00
2       1003      701    click  2026-06-09 10:40:00
3       1004      702      buy  2026-06-09 11:15:00


### 分别使用 Pandas 轨道（链式或分流流式） 和 SQL 轨道，完成以下审计。
👑 任务交付标准：输出一张按用户和状态类型坍塌的最终风控审计大盘报表。报表必须包含且仅包含以下三个核心字段，并按用户 ID 升序排列：
* user_id：用户唯一标识。
* status：剥离空格后的纯净状态名称（如 VIP_START 或 RISK_LOCK）。
* valid_behavior_count：在这个状态发生后 30 分钟内（$\ge 0s$ 且 $\le 30m$），该用户发生的有效行为总次数。

In [35]:
# ====================================================
# 🧱 SQL 轨道 - 架构师级非等值流转完全体（终极无漏洞版）
# ====================================================
sql_query = """
WITH status_log_check AS (
    SELECT 
        user_id,
        TRIM(status) AS status,
        datetime(TRIM(status_time)) AS status_time,
        CASE
            -- 👑 修正一：让上一次的身份（status）与当前身份（status）进行纯净物理比对
            WHEN LAG(TRIM(status)) OVER(PARTITION BY user_id ORDER BY datetime(TRIM(status_time)) ASC) = TRIM(status)
            THEN 1 ELSE 0
        END AS is_duplicated
    FROM status_log
),
status_log_filtered AS (
    SELECT
        user_id,
        status,
        status_time
    FROM status_log_check
    WHERE is_duplicated = 0
),
status_log_cleaned AS (
    SELECT
        user_id,
        status,
        status_time,
        COALESCE(
            LEAD(status_time) OVER(PARTITION BY user_id ORDER BY status_time ASC),
            '9999-12-31 23:59:59'
        ) AS status_end_time
    -- 👑 修正二：补上丢失的 FROM 桥梁，死死咬住上一层过滤后的纯净数据
    FROM status_log_filtered
),
behavior_log_cleaned AS (
    SELECT
        user_id,
        datetime(TRIM(behavior_time)) AS behavior_time
    FROM behavior_log
),
valid_match AS (
    SELECT
        s.user_id,
        s.status,
        b.behavior_time
    FROM status_log_cleaned AS s
    INNER JOIN behavior_log_cleaned AS b ON s.user_id = b.user_id
        -- 👑 修正三：将状态起点拨回左表 s！让行为时间（b）去跟状态时间（s）做时空合围
        AND b.behavior_time >= s.status_time
        AND b.behavior_time <= datetime(s.status_time, '+30 minutes')
        AND b.behavior_time < s.status_end_time
)
SELECT 
    user_id, 
    status, 
    COUNT(behavior_time) AS valid_behavior_count
FROM valid_match
GROUP BY user_id, status
ORDER BY user_id ASC;
"""

df_sql = pd.read_sql_query(sql_query, conn)
print("=== 👑 SQL 轨道：听取克星意见修正后，真理大盘落地 ===")
print(df_sql.to_string(index=False))

=== 👑 SQL 轨道：听取克星意见修正后，真理大盘落地 ===
 user_id    status  valid_behavior_count
     701 RISK_LOCK                     2
     701 VIP_START                     1
     702 VIP_START                     1


In [36]:
# ====================================================
# 🧱 PANDAS 轨道 - 完全体
# ====================================================
df_status['status'] = df_status['status'].str.strip()

# 🛡️ 1. 核心修正：时间列强制转为 datetime64 类型
df_status['status_time'] = pd.to_datetime(df_status['status_time'].str.strip())
df_behavior['behavior_time'] = pd.to_datetime(df_behavior['behavior_time'].str.strip())

# 2. 状态流内部筑起防爆盾（去重）
df_status = df_status.sort_values(by=['user_id', 'status_time']).reset_index(drop=True)
df_status['is_duplicated'] = (
    df_status
    .groupby('user_id')['status']
    .transform(lambda x: x.shift(1)) == df_status['status']
).astype(int)

# 🧹 3. 洗净：过滤掉重复状态，只保留真正的身份流转节点
status_log_cleaned = df_status.query("is_duplicated == 0")

# 4. ⚔️ 时空合围：此时两边已是纯净的时间轴，雷达探测完美通关
df_matched = pd.merge_asof(
    left=df_behavior.sort_values(by='behavior_time'),
    right=status_log_cleaned.sort_values(by='status_time'),
    left_on='behavior_time',
    right_on='status_time',
    by='user_id',
    direction='backward'
)

# 5. 🗜️ 弹性大闸：卡死 30 分钟生命时效
df_matched['time_diff'] = df_matched['behavior_time'] - df_matched['status_time']
df_final = df_matched.query("time_diff >= '0 days' and time_diff <= '30 minutes'")

# 6. 👑 最终收割：聚合计算大盘
df_result = (
    df_final
    .groupby(['user_id', 'status'])['action_id']
    .count()
    .reset_index(name='valid_behavior_count')
)

print(df_result)

   user_id     status  valid_behavior_count
0      701  RISK_LOCK                     2
1      701  VIP_START                     1
2      702  VIP_START                     1
